In [1]:
# %% [markdown]
# # Diagnostic 5: Theoretical Loss Baseline Test
# **Objective:** Compare step-0 loss against theoretical cross-entropy baseline to catch initialization bugs.

# %%
import torch
import torch.nn as nn
import math

torch.manual_seed(42)


In [2]:

NUM_CLASSES = 10
IN_FEATURES = 16
BATCH_SIZE = 1024

# 1. 2-layer network outputting unnormalized logits
class InitializedNet(nn.Module):
    def __init__(self, in_features, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

model = InitializedNet(in_features=IN_FEATURES, hidden_dim=32, num_classes=NUM_CLASSES)
criterion = nn.CrossEntropyLoss()


In [3]:

# 2. Balanced uniform dummy batch
X = torch.randn(BATCH_SIZE, IN_FEATURES)
# Ensure uniform distribution across classes
y = torch.randint(0, NUM_CLASSES, (BATCH_SIZE,))


In [4]:

# 3. Calculate initial step-0 empirical loss
with torch.no_grad():
    logits = model(X)
    empirical_loss = criterion(logits, y).item()

# 4. Compute theoretical loss baseline: -ln(1 / C)
theoretical_loss = -math.log(1.0 / NUM_CLASSES)
relative_error = abs(empirical_loss - theoretical_loss) / theoretical_loss

print(f"Number of balanced classes (C): {NUM_CLASSES}")
print(f"Theoretical Initial Loss:        {theoretical_loss:.5f}")
print(f"Observed Step-0 Loss:            {empirical_loss:.5f}")
print(f"Relative Deviation:              {relative_error * 100:.2f}%")

# 5. Diagnostic evaluation
print("\n--- Diagnostic Verdict ---")
if relative_error < 0.10:
    print("PASSED: Initial loss matches theoretical random baseline within 10% tolerance.")
else:
    print("FAILED: Substantial baseline deviation. Inspect output activations (e.g. redundant softmax) or weight scale.")

Number of balanced classes (C): 10
Theoretical Initial Loss:        2.30259
Observed Step-0 Loss:            2.31577
Relative Deviation:              0.57%

--- Diagnostic Verdict ---
PASSED: Initial loss matches theoretical random baseline within 10% tolerance.


# Diagnostic 5b: What Does the Initial-Loss Test Actually Catch?

The test compares step-0 loss with `ln(C)`, the loss of a model that predicts every class with probability `1/C`. It can only see problems that move the **step-0 loss away from `ln(C)`**. This section injects eight errors, one at a time, into the same model and batch (same seed, same `InitializedNet`, same 10% tolerance), then reports which ones it finds and which it cannot.

| Group | # | Injected error | Expected |
|---|---|---|---|
| **Should be caught** | 1 | Final layer too large (`fc2.weight * 10`) | Loss far above `ln C` |
| | 2 | Unscaled inputs (`X * 100`) | Loss far above `ln C` |
| | 3 | `reduction='sum'` instead of `'mean'` | Loss about `batch_size` times too large |
| | 4 | Wrong output dimension (100 logits, 10 classes) | Loss about `ln 100` |
| **Should be missed** | 5 | Double softmax (`softmax` before `CrossEntropyLoss`) | Loss pinned at `ln C` |
| | 6 | Class imbalance (about 90% one class) | Uniform predictions give `ln C` whatever the labels |
| | 7 | Shuffled (misaligned) labels | Random labels also give `ln C` |
| | 8 | Dead hidden layer (`fc1.bias = -10`) | Logits are constant, loss stays near `ln C` |

The failure message in the cell above suggests inspecting a "redundant softmax". The results below show why that hint is misleading: a redundant softmax does not make this test fail.

In [5]:
import torch.nn.functional as F

TOLERANCE = 0.10                       # same rule as the original verdict
theoretical_loss = -math.log(1.0 / NUM_CLASSES)

def make(seed, num_outputs=NUM_CLASSES):
    """Same construction order as the original: seed, model, balanced batch."""
    torch.manual_seed(seed)
    model = InitializedNet(in_features=IN_FEATURES, hidden_dim=32, num_classes=num_outputs)
    X = torch.randn(BATCH_SIZE, IN_FEATURES)
    y = torch.randint(0, NUM_CLASSES, (BATCH_SIZE,))
    return model, X, y

# Each scenario returns the step-0 loss for a given seed.
def s_control(seed):
    model, X, y = make(seed)
    with torch.no_grad():
        return nn.CrossEntropyLoss()(model(X), y).item()

def s_big_last_layer(seed):
    model, X, y = make(seed)
    with torch.no_grad():
        model.fc2.weight.mul_(10.0)
        return nn.CrossEntropyLoss()(model(X), y).item()

def s_unscaled_inputs(seed):
    model, X, y = make(seed)
    with torch.no_grad():
        return nn.CrossEntropyLoss()(model(X * 100.0), y).item()

def s_reduction_sum(seed):
    model, X, y = make(seed)
    with torch.no_grad():
        return nn.CrossEntropyLoss(reduction="sum")(model(X), y).item()

def s_wrong_out_dim(seed):
    model, X, y = make(seed, num_outputs=100)       # 100 logits, labels still in 0..9
    with torch.no_grad():
        return nn.CrossEntropyLoss()(model(X), y).item()

def s_double_softmax(seed):
    model, X, y = make(seed)
    with torch.no_grad():
        return nn.CrossEntropyLoss()(torch.softmax(model(X), dim=1), y).item()

def s_imbalanced(seed):
    model, X, y = make(seed)
    y = torch.where(torch.rand(BATCH_SIZE) < 0.9, torch.zeros_like(y), y)   # about 90% class 0
    with torch.no_grad():
        return nn.CrossEntropyLoss()(model(X), y).item()

def s_shuffled_labels(seed):
    model, X, y = make(seed)
    with torch.no_grad():
        return nn.CrossEntropyLoss()(model(X), y[torch.randperm(BATCH_SIZE)]).item()

def s_dead_hidden(seed):
    model, X, y = make(seed)
    with torch.no_grad():
        model.fc1.bias.fill_(-10.0)                 # every hidden ReLU outputs 0
        return nn.CrossEntropyLoss()(model(X), y).item()

results = []

def check(label, fn, injected=True, seed=42):
    """Original test output and verdict, applied to one scenario."""
    empirical_loss = fn(seed)
    relative_error = abs(empirical_loss - theoretical_loss) / theoretical_loss
    flagged = relative_error >= TOLERANCE
    print(f"=== {label} ===")
    print(f"Theoretical Initial Loss:        {theoretical_loss:.5f}")
    print(f"Observed Step-0 Loss:            {empirical_loss:.5f}")
    print(f"Relative Deviation:              {relative_error * 100:.2f}%")
    if not flagged:
        print("Verdict: PASSED: Initial loss matches theoretical random baseline within 10% tolerance.")
    else:
        print("Verdict: FAILED: Substantial baseline deviation.")
    print()
    results.append(dict(label=label, fn=fn, injected=injected, loss=empirical_loss,
                        rel=relative_error, flagged=flagged))

In [6]:
# Control: no injected error (reproduces the original numbers)
check("Control: correct setup", s_control, injected=False)

=== Control: correct setup ===
Theoretical Initial Loss:        2.30259
Observed Step-0 Loss:            2.31577
Relative Deviation:              0.57%
Verdict: PASSED: Initial loss matches theoretical random baseline within 10% tolerance.



In [7]:
# ERROR 1 (should be caught): final layer initialized 10x too large
check("Error 1: fc2.weight * 10", s_big_last_layer)

=== Error 1: fc2.weight * 10 ===
Theoretical Initial Loss:        2.30259
Observed Step-0 Loss:            4.07209
Relative Deviation:              76.85%
Verdict: FAILED: Substantial baseline deviation.



In [8]:
# ERROR 2 (should be caught): inputs not standardized
check("Error 2: unscaled inputs (X * 100)", s_unscaled_inputs)

=== Error 2: unscaled inputs (X * 100) ===
Theoretical Initial Loss:        2.30259
Observed Step-0 Loss:            33.62701
Relative Deviation:              1360.40%
Verdict: FAILED: Substantial baseline deviation.



In [9]:
# ERROR 3 (should be caught): loss summed over the batch instead of averaged
check("Error 3: reduction='sum'", s_reduction_sum)

=== Error 3: reduction='sum' ===
Theoretical Initial Loss:        2.30259
Observed Step-0 Loss:            2371.34912
Relative Deviation:              102886.38%
Verdict: FAILED: Substantial baseline deviation.



In [10]:
# ERROR 4 (should be caught): output layer has 100 units for a 10-class problem
check("Error 4: 100 output units for 10 classes", s_wrong_out_dim)

=== Error 4: 100 output units for 10 classes ===
Theoretical Initial Loss:        2.30259
Observed Step-0 Loss:            4.68996
Relative Deviation:              103.68%
Verdict: FAILED: Substantial baseline deviation.



In [11]:
# ERROR 5 (should be missed): softmax applied before CrossEntropyLoss
check("Error 5: double softmax", s_double_softmax)

=== Error 5: double softmax ===
Theoretical Initial Loss:        2.30259
Observed Step-0 Loss:            2.30178
Relative Deviation:              0.04%
Verdict: PASSED: Initial loss matches theoretical random baseline within 10% tolerance.



In [12]:
# ERROR 6 (should be missed): heavily imbalanced labels
check("Error 6: class imbalance (~90% one class)", s_imbalanced)

=== Error 6: class imbalance (~90% one class) ===
Theoretical Initial Loss:        2.30259
Observed Step-0 Loss:            2.38845
Relative Deviation:              3.73%
Verdict: PASSED: Initial loss matches theoretical random baseline within 10% tolerance.



In [13]:
# ERROR 7 (should be missed): labels shuffled relative to inputs
check("Error 7: shuffled / misaligned labels", s_shuffled_labels)

=== Error 7: shuffled / misaligned labels ===
Theoretical Initial Loss:        2.30259
Observed Step-0 Loss:            2.33283
Relative Deviation:              1.31%
Verdict: PASSED: Initial loss matches theoretical random baseline within 10% tolerance.



In [14]:
# ERROR 8 (should be missed): dead hidden layer
check("Error 8: dead hidden layer (fc1.bias = -10)", s_dead_hidden)

=== Error 8: dead hidden layer (fc1.bias = -10) ===
Theoretical Initial Loss:        2.30259
Observed Step-0 Loss:            2.30467
Relative Deviation:              0.09%
Verdict: PASSED: Initial loss matches theoretical random baseline within 10% tolerance.



In [15]:
# Summary, plus a 20-seed check so no conclusion rests on seed 42 alone
SEEDS = range(20)
print(f"{'Scenario':<46}{'Loss (seed 42)':>15}{'Dev %':>9}  {'Verdict':<8}{'20-seed mean':>14}{'% seeds flagged':>17}")
print("-" * 112)
for r in results:
    losses = torch.tensor([r["fn"](s) for s in SEEDS])
    pct_flagged = ((losses - theoretical_loss).abs() / theoretical_loss >= TOLERANCE).float().mean().item() * 100
    verdict = "FAILED" if r["flagged"] else "PASSED"
    print(f"{r['label']:<46}{r['loss']:>15.4f}{r['rel'] * 100:>9.1f}  {verdict:<8}{losses.mean().item():>14.4f}{pct_flagged:>16.0f}%")
    r["pct_flagged"] = pct_flagged

errors = [r for r in results if r["injected"]]
found = [r for r in errors if r["flagged"]]
print()
print(f"Injected errors found by the test: {len(found)} of {len(errors)}")
print("Found: ", "; ".join(r["label"] for r in found))
print("Missed:", "; ".join(r["label"] for r in errors if not r["flagged"]))

Scenario                                       Loss (seed 42)    Dev %  Verdict   20-seed mean  % seeds flagged
----------------------------------------------------------------------------------------------------------------
Control: correct setup                                 2.3158      0.6  PASSED          2.3346               0%
Error 1: fc2.weight * 10                               4.0721     76.8  FAILED          4.2168             100%
Error 2: unscaled inputs (X * 100)                    33.6270   1360.4  FAILED         34.6432             100%
Error 3: reduction='sum'                            2371.3491 102886.4  FAILED       2390.6238             100%


Error 4: 100 output units for 10 classes               4.6900    103.7  FAILED          4.6369             100%


Error 5: double softmax                                2.3018      0.0  PASSED          2.3030               0%


Error 6: class imbalance (~90% one class)              2.3884      3.7  PASSED          2.3702              25%


Error 7: shuffled / misaligned labels                  2.3328      1.3  PASSED          2.3332               0%
Error 8: dead hidden layer (fc1.bias = -10)            2.3047      0.1  PASSED          2.3070               0%

Injected errors found by the test: 4 of 8
Found:  Error 1: fc2.weight * 10; Error 2: unscaled inputs (X * 100); Error 3: reduction='sum'; Error 4: 100 output units for 10 classes
Missed: Error 5: double softmax; Error 6: class imbalance (~90% one class); Error 7: shuffled / misaligned labels; Error 8: dead hidden layer (fc1.bias = -10)


In [16]:
# How large must a scale error be before the 10% tolerance flags it? Sweep the final-layer scale.
print(f"{'fc2 scale':>10}{'mean loss (20 seeds)':>22}{'deviation %':>14}  Verdict")
for scale in [1, 2, 3, 4, 5, 10]:
    losses = []
    for s in range(20):
        model, X, y = make(s)
        with torch.no_grad():
            model.fc2.weight.mul_(scale)
            model.fc2.bias.mul_(scale)
            losses.append(nn.CrossEntropyLoss()(model(X), y).item())
    mean_loss = sum(losses) / len(losses)
    dev = abs(mean_loss - theoretical_loss) / theoretical_loss
    print(f"{scale:>10}{mean_loss:>22.4f}{dev * 100:>14.1f}  {'FAILED' if dev >= TOLERANCE else 'PASSED'}")

 fc2 scale  mean loss (20 seeds)   deviation %  Verdict
         1                2.3346           1.4  PASSED
         2                2.4260           5.4  PASSED
         3                2.5718          11.7  FAILED


         4                2.7643          20.1  FAILED
         5                2.9950          30.1  FAILED


        10                4.4938          95.2  FAILED


## Takeaways

- **What it catches:** anything that pushes the step-0 loss away from `ln(C)`: an oversized final layer or unscaled inputs (logit scale too large), a loss that is off by a constant factor such as `reduction='sum'`, and an output dimension that does not match the number of classes (the loss reads `ln` of the wrong number).
- **What it cannot see, and why:**
  - **Double softmax** squeezes the logits into [0, 1], which *pins* the loss at `ln(C)`. The test passes, and the bug only shows up as a plateau during training, so use the single-batch overfit test for it.
  - **Class imbalance** does not change the loss of a uniform predictor, so the mean step-0 loss stays near `ln(C)` (2.37 over 20 seeds). It does make the loss noisier: with about 90% of labels in one class, how that class's logit happens to be initialized swings the loss, and the check flagged it in 5 of 20 seeds (25%) by chance. That is noise, not a reliable detection. The meaningful baseline for imbalance is the label entropy, which is lower than `ln(C)`.
  - **Misaligned labels** also give `ln(C)`, because random labels are what a uniform predictor already expects.
  - **A dead hidden layer** leaves constant logits near zero, so the loss stays near `ln(C)`. The activation-distribution test catches it.
- **The failure message above is misleading.** It tells you to inspect a redundant softmax, but that is a case this test passes. A failure here points to scale, reduction or output-size problems.
- **Sensitivity:** with the 10% tolerance the test flags the final-layer scale at about 3x its default init and above (see the sweep). Smaller scale errors pass, so do not read a pass as proof that the scale is right.
- **Rule of thumb for small excess:** `loss ≈ ln(C) + σ²(1 − 1/C) / 2`, where σ is the standard deviation of the logits. You can back out the logit scale from how far the loss sits above `ln(C)`.
- **Use a large enough batch.** With a handful of samples the step-0 loss can land below `ln(C)` by chance. This test uses 1024.